# 流程二 · 感知模型

训练单导联、双导联两个因果 1D-ResNet（逐秒输出）和 12 导联 Medformer（G5，整段 10 秒），结果写到 `2_Output/`。ResNet 从原始波形做单向因果滤波，只用当前秒以前的信号；Medformer 用流程一的 `X`。需先运行 `1.ipynb`，在项目根目录 Run All（各格按顺序依赖）。

## 1. 读取输入与对齐

读取流程一的文件，按 `ecg_id` 核对病人和官方分折；选择 CUDA / MPS / CPU，设置训练参数。

In [1]:
import hashlib
import json
import math
import time
from contextlib import nullcontext
from pathlib import Path
from pickle import UnpicklingError
from tempfile import NamedTemporaryFile, TemporaryDirectory

import numpy as np
import pandas as pd
import torch
import wfdb
from scipy.signal import butter, sosfilt, sosfiltfilt
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.model_selection import GroupKFold
from torch import nn
from torch.nn import functional as F

PROJECT_DIR = Path.cwd()
INPUT_DIR = PROJECT_DIR / "1_Output"
OUTPUT_DIR = PROJECT_DIR / "2_Output"
DATA_DIR = PROJECT_DIR / "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
SPLITS = ("train", "val", "test")
CLASSES = ["NORM", "AFIB", "MI", "STTC"]
SEED = 42
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
if device.type == "cuda":
    # Keep batch-size-dependent TF32 rounding out of comparisons with prefix replay.
    torch.backends.cudnn.allow_tf32 = False
    torch.backends.cuda.matmul.allow_tf32 = False
if device.type != "cpu":
    torch.set_num_threads(min(8, torch.get_num_threads()))

# Larger batches increase peak accelerator memory. Change these if needed on your GPU.
RESNET_OPTIONS = dict(
    kernel_size=11, windows_per_record=3, max_epochs=50, patience=9, lr_patience=3,
    batch_size=256, eval_batch_size=512, validation_steps=tuple(range(1, 11)),
    learning_rate=1e-3, cuda_bfloat16=True, seed=SEED,
)
MEDFORMER_OPTIONS = dict(
    max_epochs=60, patience=12, batch_size=96, learning_rate=3e-4, seed=SEED,
    model_kwargs=dict(patch_lengths=[20, 50, 100], width=128, layers=3, heads=8, dropout=0.15),
)

phase1 = json.loads((INPUT_DIR / "preprocessing.json").read_text(encoding="utf-8"))
if phase1["class_order"] != CLASSES:
    raise ValueError("Unexpected Phase 1 class order")
patient_map = pd.read_csv(INPUT_DIR / "patient_map.csv").set_index("ecg_id", verify_integrity=True)
offline_inputs, labels, ids, patient_ids = {}, {}, {}, {}
for split in SPLITS:
    ids[split] = pd.read_csv(INPUT_DIR / f"meta_{split}.csv")["ecg_id"].to_numpy()
    patient_ids[split] = patient_map.loc[ids[split], "patient_id"].to_numpy()
    offline_inputs[split] = np.load(INPUT_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
    if offline_inputs[split].shape != (len(ids[split]), 12, 1000):
        raise ValueError(f"Unexpected waveform shape in {split}")
    if split in ("train", "val"):
        one_hot = np.load(INPUT_DIR / f"y_{split}.npy", allow_pickle=False)
        if one_hot.shape != (len(ids[split]), 4) or not np.all((one_hot == 0) | (one_hot == 1)) or not np.all(one_hot.sum(axis=1) == 1):
            raise ValueError(f"Invalid one-hot labels in {split}")
        labels[split] = one_hot.argmax(axis=1)

if sum(len(set(values)) for values in patient_ids.values()) != len(set(np.concatenate(list(patient_ids.values())))):
    raise ValueError("Patient IDs overlap between the official splits")

print(f"Training device: {device}; PyTorch {torch.__version__}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
print("Training output:", OUTPUT_DIR)
print({split: len(ids[split]) for split in SPLITS})

Training device: cpu; PyTorch 2.14.1+cpu
Training output: E:\下载\qq\AI_Project.zip\AI_Project\2_Output
{'train': 15259, 'val': 1891, 'test': 1914}


In [2]:
database = pd.read_csv(DATA_DIR / "ptbxl_database.csv", usecols=[
    "ecg_id", "patient_id", "strat_fold", "filename_lr"
]).set_index("ecg_id", verify_integrity=True)
official_folds = {"train": set(range(1, 9)), "val": {9}, "test": {10}}
all_ids = np.concatenate(list(ids.values()))
if len(np.unique(all_ids)) != len(all_ids) or set(all_ids) != set(patient_map.index):
    raise ValueError("ECG IDs must be unique across splits and match the Phase 1 patient map")
for split in SPLITS:
    aligned = database.loc[ids[split]]
    np.testing.assert_array_equal(aligned.patient_id.to_numpy(), patient_ids[split])
    if set(aligned.strat_fold) != official_folds[split]:
        raise ValueError(f"Official patient folds do not match in {split}")
    for ecg_id, name in aligned.filename_lr.items():
        path = Path(name)
        if path.parts[0] != "records100" or path.name != f"{ecg_id:05d}_lr":
            raise ValueError(f"Unexpected low-rate waveform path for ECG {ecg_id}")
print(f"Matched {len(all_ids):,} ECGs to raw recordings and official patient folds.")

Matched 19,064 ECGs to raw recordings and official patient folds.


## 2. 构建流式输入

Lead I 和 V1 做 0.5–40 Hz 单向因果滤波，用训练集均值和标准差标准化；切出在第 1–10 秒结束的 3 秒窗口，历史不足时左边补零。V1 在第 w 秒末唤醒时，从唤醒时刻开始滤波。

In [3]:
STREAM_SAMPLE_RATE = 100
STREAM_DURATION = 10
STREAM_SAMPLES = STREAM_SAMPLE_RATE * STREAM_DURATION
STREAM_LEADS = ("I", "V1")
STREAM_SPLITS = ("train", "val", "test")
STREAM_BAND = (0.5, 40.0)
STREAM_FILTER_ORDER = 4
STREAM_STD_EPSILON = 1e-6
STREAM_SOS = butter(
    STREAM_FILTER_ORDER, STREAM_BAND, fs=STREAM_SAMPLE_RATE, btype="bandpass", output="sos"
)


def read_stream_record(record_path):
    """Read the two physical WFDB leads in I, V1 order."""
    waveform, info = wfdb.rdsamp(str(record_path), channel_names=list(STREAM_LEADS))
    if info["fs"] != STREAM_SAMPLE_RATE or waveform.shape != (STREAM_SAMPLES, len(STREAM_LEADS)):
        raise ValueError(f"Unexpected sample rate or shape in {record_path}")
    if tuple(info["sig_name"]) != STREAM_LEADS or any(unit != "mV" for unit in info["units"]):
        raise ValueError(f"Unexpected lead order or voltage unit in {record_path}")
    if not np.isfinite(waveform).all():
        raise ValueError(f"Non-finite voltage in {record_path}")
    return waveform.T


def filter_stream(raw, sos=STREAM_SOS):
    """Run a one-way filter starting at the first recorded sample, with zero initial state."""
    return sosfilt(sos, raw, axis=-1)


def fit_stream_statistics(train):
    """Fit per-lead population moments on the causal training recordings."""
    total = np.zeros(len(STREAM_LEADS), dtype=np.float64)
    total_squared = np.zeros_like(total)
    for start in range(0, len(train), 256):
        chunk = train[start : start + 256].astype(np.float64)
        total += chunk.sum(axis=(0, 2))
        total_squared += np.square(chunk).sum(axis=(0, 2))
    count = len(train) * STREAM_SAMPLES
    if count == 0:
        raise ValueError("Cannot fit stream normalization without training ECGs")
    mean = total / count
    standard_deviation = np.sqrt(np.maximum(total_squared / count - np.square(mean), 0.0))
    return {
        "lead_order": list(STREAM_LEADS),
        "sample_rate_hz": STREAM_SAMPLE_RATE,
        "samples_per_record": STREAM_SAMPLES,
        "signal_input_units": "mV",
        "filter": "scipy.signal.sosfilt",
        "filter_initial_state": "zero at the first observed sample of each lead",
        "bandpass_hz": list(STREAM_BAND),
        "butterworth_order": STREAM_FILTER_ORDER,
        "sos": STREAM_SOS.tolist(),
        "normalization": "one population mean and standard deviation per lead, fitted on selected training records only",
        "fit_beginning": "both leads observed from the first sample",
        "fit_records": len(train),
        "fit_samples_per_lead": count,
        "mean_mv": mean.tolist(),
        "std_mv": standard_deviation.tolist(),
        "std_epsilon": STREAM_STD_EPSILON,
        "window_padding": "zero in standardized units, applied only to unavailable samples",
        "wake_convention": "a V1 wake action at the end of step w makes samples from index w * sample_rate_hz available",
    }


def standardize_stream(filtered, preprocessing):
    """Apply the frozen normalization; filtered input has trailing axes (lead, sample)."""
    mean = np.asarray(preprocessing["mean_mv"], dtype=np.float64)[:, None]
    scale = np.asarray(preprocessing["std_mv"], dtype=np.float64)[:, None] + preprocessing["std_epsilon"]
    return ((filtered - mean) / scale).astype(np.float32)


def build_stream_inputs():
    """Return split streams, raw V1 replays and fitted preprocessing for Phase 2."""
    paths = pd.read_csv(DATA_DIR / "ptbxl_database.csv", usecols=["ecg_id", "filename_lr"]).set_index("ecg_id")
    if not paths.index.is_unique:
        raise ValueError("Raw ECG IDs must be unique")

    streams = {}
    raw_v1 = {}
    preprocessing = None
    for split in STREAM_SPLITS:
        ids = pd.read_csv(INPUT_DIR / f"meta_{split}.csv", usecols=["ecg_id"])["ecg_id"]
        if not ids.is_unique:
            raise ValueError(f"Duplicate ECG IDs in the {split} metadata")
        relative_paths = paths.loc[ids, "filename_lr"]
        filtered = np.empty((len(ids), len(STREAM_LEADS), STREAM_SAMPLES), dtype=np.float32)
        raw_v1[split] = np.empty((len(ids), STREAM_SAMPLES), dtype=np.float32)

        for start in range(0, len(ids), 256):
            batch_paths = relative_paths.iloc[start : start + 256]
            raw = np.empty((len(batch_paths), len(STREAM_LEADS), STREAM_SAMPLES), dtype=np.float32)
            for row, relative_path in enumerate(batch_paths):
                raw[row] = read_stream_record(DATA_DIR / relative_path)
            stop = start + len(batch_paths)
            raw_v1[split][start:stop] = raw[:, 1]
            filtered[start:stop] = filter_stream(raw)

        if split == "train":
            preprocessing = fit_stream_statistics(filtered)
        for start in range(0, len(filtered), 256):
            filtered[start : start + 256] = standardize_stream(filtered[start : start + 256], preprocessing)
        streams[split] = filtered

    validate_stream_causality(preprocessing)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    stats_path = OUTPUT_DIR / "stream_preprocessing.json"
    stats_path.write_text(json.dumps(preprocessing, indent=2, allow_nan=False) + "\n", encoding="utf-8")
    return streams, raw_v1, preprocessing

In [4]:
def stream_steps(steps, batch_size):
    """Expand a scalar step or validate one integer step per sample."""
    steps = np.asarray(steps)
    if not np.issubdtype(steps.dtype, np.integer):
        raise ValueError("Observation and wake steps must use integer indices")
    if steps.ndim == 0:
        return np.full(batch_size, steps, dtype=np.int64)
    if steps.shape != (batch_size,):
        raise ValueError("Pass one step for the batch or one step per recording")
    return steps.astype(np.int64, copy=False)


def make_causal_windows(signals, end_steps, window_seconds=3):
    """Extract windows ending at the specified observed steps, with left padding."""
    if signals.ndim != 3 or signals.shape[-1] == 0 or signals.shape[-1] % STREAM_SAMPLE_RATE != 0:
        raise ValueError("Signals must have shape (batch, lead, whole seconds of samples)")
    if not isinstance(window_seconds, (int, np.integer)) or window_seconds <= 0:
        raise ValueError("Window duration must be a positive number of whole seconds")
    steps = stream_steps(end_steps, len(signals))
    end_indices = steps * STREAM_SAMPLE_RATE
    if np.any(steps < 1) or np.any(end_indices > signals.shape[-1]):
        raise ValueError("Observation step outside the available data")

    offsets = np.arange(-window_seconds * STREAM_SAMPLE_RATE, 0)
    positions = end_indices[:, None, None] + offsets
    windows = np.take_along_axis(signals, np.maximum(positions, 0), axis=-1)
    return np.where(positions < 0, 0, windows)


def make_wake_windows(lead_i, raw_v1, wake_steps, end_steps, preprocessing):
    """Return [batch, 2, 300] after replaying V1 from each record's wake point.

    Lead I is already causal and normalized. A wake at end step w gives V1
    samples for step w+1 onward; w=0 represents V1 active from the beginning.
    Replaying from wake remains necessary after V1 has a full window because
    a one-way IIR filter retains state older than the window.
    """
    lead_i = np.asarray(lead_i)
    raw_v1 = np.asarray(raw_v1)
    if lead_i.ndim != 2 or lead_i.shape != raw_v1.shape or lead_i.shape[1] != STREAM_SAMPLES:
        raise ValueError("Lead I and V1 must have matching (batch, 1000) shapes")
    wakes = stream_steps(wake_steps, len(lead_i))
    ends = stream_steps(end_steps, len(lead_i))
    if np.any(wakes < 0) or np.any(wakes >= ends) or np.any(ends > STREAM_DURATION):
        raise ValueError("Wake steps must precede their observation steps and fall within the record")

    lead_i_windows = make_causal_windows(lead_i[:, None, :], ends)
    windows = np.zeros((len(lead_i), 2, lead_i_windows.shape[-1]), dtype=np.float32)
    windows[:, 0] = lead_i_windows[:, 0]
    sos = np.asarray(preprocessing["sos"])
    mean = preprocessing["mean_mv"][1]
    scale = preprocessing["std_mv"][1] + preprocessing["std_epsilon"]

    for wake in np.unique(wakes):
        rows = np.flatnonzero(wakes == wake)
        observed_steps = ends[rows] - wake
        start = int(wake) * STREAM_SAMPLE_RATE
        stop = int(ends[rows].max()) * STREAM_SAMPLE_RATE
        causal_v1 = filter_stream(raw_v1[rows, start:stop], sos).astype(np.float32)
        causal_v1 = ((causal_v1.astype(np.float64) - mean) / scale).astype(np.float32)
        windows[rows, 1] = make_causal_windows(causal_v1[:, None, :], observed_steps)[:, 0]
    return windows


def validate_stream_causality(preprocessing):
    """Check that modifying future samples or inactive V1 cannot change the current window."""
    random = np.random.default_rng(19)
    raw = random.normal(0, 0.2, size=(2, len(STREAM_LEADS), STREAM_SAMPLES)).astype(np.float32)
    filtered = standardize_stream(filter_stream(raw).astype(np.float32), preprocessing)

    changed = raw.copy()
    changed[:, :, 400:] += 0.7
    changed_filtered = standardize_stream(filter_stream(changed).astype(np.float32), preprocessing)
    np.testing.assert_array_equal(
        make_causal_windows(filtered, 4),
        make_causal_windows(changed_filtered, 4),
        err_msg="Causal windows changed when future voltage changed",
    )

    changed = raw[:, 1].copy()
    changed[:, :400] += 0.7
    changed[:, 500:] -= 0.7
    post_wake = make_wake_windows(filtered[:, 0], raw[:, 1], 4, 5, preprocessing)
    np.testing.assert_array_equal(
        post_wake,
        make_wake_windows(filtered[:, 0], changed, 4, 5, preprocessing),
        err_msg="Wake windows changed when unavailable V1 voltage changed",
    )
    np.testing.assert_array_equal(post_wake[:, 1, :200], 0.0)
    np.testing.assert_array_equal(make_wake_windows(filtered[:, 0], raw[:, 1], 0, 5, preprocessing),
                                  make_causal_windows(filtered, 5))

## 3. 因果 1D-ResNet-18

4 个 stage、8 个残差块，通道 16 / 32 / 64 / 128，卷积核 11；单导联分支输入 Lead I，双导联分支输入 Lead I + V1。

In [5]:
RESNET_STEPS = 10
RESNET_RATE = 100
RESNET_WINDOW = 300
RESNET_CACHE_VERSION = 3


class CausalConv1d(nn.Conv1d):
    """Left-pad and align strided outputs to the latest input sample."""

    def forward(self, x):
        offset = (x.shape[-1] - 1) % self.stride[0]
        left = self.kernel_size[0] - 1 - offset
        if left > 0:
            x = F.pad(x, (left, 0))
        elif left < 0:
            x = x[..., -left:]
        return super().forward(x)


class ResNetBlock1d(nn.Module):
    def __init__(self, input_width, width, stride, kernel_size):
        super().__init__()
        self.conv1 = CausalConv1d(input_width, width, kernel_size, stride=stride, bias=False)
        self.bn1 = nn.BatchNorm1d(width)
        self.conv2 = CausalConv1d(width, width, kernel_size, bias=False)
        self.bn2 = nn.BatchNorm1d(width)
        self.projection = (
            nn.Sequential(
                CausalConv1d(input_width, width, 1, stride=stride, bias=False),
                nn.BatchNorm1d(width),
            )
            if stride != 1 or input_width != width
            else nn.Identity()
        )

    def forward(self, x):
        residual = self.projection(x)
        x = F.relu(self.bn1(self.conv1(x)), inplace=True)
        x = self.bn2(self.conv2(x))
        return F.relu(x + residual, inplace=True)


class CausalResNet18(nn.Module):
    def __init__(self, input_channels, kernel_size=11):
        super().__init__()
        if kernel_size not in (7, 9, 11, 13, 15):
            raise ValueError("kernel_size must be an odd integer from 7 through 15")
        self.stem = nn.Sequential(
            CausalConv1d(input_channels, 16, kernel_size, stride=2, bias=False),
            nn.BatchNorm1d(16),
            nn.ReLU(inplace=True),
        )
        widths = (16, 32, 64, 128)
        blocks = []
        previous_width = widths[0]
        for stage, width in enumerate(widths):
            blocks.append(ResNetBlock1d(previous_width, width, 1 if stage == 0 else 2, kernel_size))
            blocks.append(ResNetBlock1d(width, width, 1, kernel_size))
            previous_width = width
        self.blocks = nn.Sequential(*blocks)
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(widths[-1], 4)

    def forward(self, x):
        x = self.stem(x)
        offset = (x.shape[-1] - 1) % 2
        x = F.max_pool1d(F.pad(x, (2 - offset, 0), value=float("-inf")), 3, stride=2)
        return self.head(self.pool(self.blocks(x)).squeeze(-1))

In [6]:
def _resnet_windows(signals):
    padded = np.pad(signals, ((0, 0), (0, 0), (RESNET_WINDOW - RESNET_RATE, 0)))
    windows = np.lib.stride_tricks.sliding_window_view(padded, RESNET_WINDOW, axis=-1)
    return windows[:, :, ::RESNET_RATE, :].transpose(0, 2, 1, 3)


def _resnet_fold_scale(train_signals, fit_rows, preprocessing=None):
    sums = np.zeros(2, dtype=np.float64)
    sum_squares = np.zeros_like(sums)
    for start in range(0, len(fit_rows), 1024):
        batch = train_signals[fit_rows[start:start + 1024]]
        sums += batch.sum(axis=(0, 2), dtype=np.float64)
        sum_squares += np.einsum("nct,nct->c", batch, batch, dtype=np.float64)
    count = len(fit_rows) * RESNET_STEPS * RESNET_RATE
    mean = sums / count
    scale = np.sqrt(np.maximum(sum_squares / count - mean**2, 0.0))
    if preprocessing is None:
        scale = np.maximum(scale, 1e-6)
    else:
        eps_mv = preprocessing["std_epsilon"]
        global_scale_mv = np.asarray(preprocessing["std_mv"]) + eps_mv
        scale += eps_mv / global_scale_mv
    return mean.astype(np.float32), scale.astype(np.float32)


def _resnet_apply_fold_scale(batch, real_seconds, fold_scale):
    if fold_scale is not None:
        mean, std = fold_scale
        channels = batch.shape[1]
        batch -= mean[None, :channels, None]
        batch /= std[None, :channels, None]
        left = RESNET_WINDOW - RESNET_RATE * real_seconds
        is_padding = np.arange(RESNET_WINDOW)[None, None, :] < left[:, :, None]
        np.copyto(batch, np.float32(0), where=is_padding)
    return batch


def _resnet_batch(windows, rows, steps, channels, fold_scale, wake_source=None, rng=None):
    batch = windows[rows, steps, :channels]
    real_seconds = np.minimum(steps + 1, 3)
    real_seconds = np.repeat(real_seconds[:, None], channels, axis=1)
    if wake_source is not None:
        choose_wake = (steps > 0) & (rng.random(len(rows)) < wake_source["fraction"])
        if choose_wake.any():
            selected_rows = rows[choose_wake]
            end_steps = steps[choose_wake] + 1
            histories = rng.integers(1, np.minimum(end_steps - 1, 2) + 1)
            wake_steps = end_steps - histories
            wake_windows = wake_source["builder"](
                wake_source["lead_i"][selected_rows],
                wake_source["raw_v1"][selected_rows],
                wake_steps,
                end_steps,
                wake_source["preprocessing"],
            )
            batch[choose_wake] = wake_windows
            real_seconds[choose_wake, 1] = histories
    return _resnet_apply_fold_scale(batch, real_seconds, fold_scale)

In [7]:
def _resnet_eval_batches(model, windows, rows, steps, channels, fold_scale, device, batch_size):
    for start in range(0, len(rows) * len(steps), batch_size):
        offsets = np.arange(start, min(start + batch_size, len(rows) * len(steps)))
        row_batch = rows[offsets // len(steps)]
        step_batch = steps[offsets % len(steps)]
        batch = _resnet_batch(windows, row_batch, step_batch, channels, fold_scale)
        logits = model(torch.from_numpy(batch).to(device))
        yield start, row_batch, logits


@torch.inference_mode()
def _resnet_validation_loss(model, windows, labels, steps, channels, fold_scale, device, batch_size):
    model.eval()
    loss = torch.zeros((), device=device)
    rows = np.arange(len(labels))
    for _, row_batch, logits in _resnet_eval_batches(
        model, windows, rows, steps, channels, fold_scale, device, batch_size
    ):
        truth = torch.from_numpy(labels[row_batch]).to(device)
        loss += F.cross_entropy(logits, truth, reduction="sum")
    return float(loss.item()) / (len(rows) * len(steps))


@torch.inference_mode()
def _resnet_probabilities(model, windows, rows, channels, fold_scale, device, batch_size):
    model.eval()
    result = np.empty((len(rows) * RESNET_STEPS, 4), dtype=np.float32)
    steps = np.arange(RESNET_STEPS)
    for start, _, logits in _resnet_eval_batches(
        model, windows, rows, steps, channels, fold_scale, device, batch_size
    ):
        probabilities = logits.softmax(dim=-1).cpu().numpy()
        result[start:start + len(probabilities)] = probabilities
    if not np.isfinite(result).all():
        raise RuntimeError("ResNet inference produced a non-finite probability")
    return result.reshape(len(rows), RESNET_STEPS, 4)

### 按病人分组的 5 折训练

训练集按病人分 5 折做 OOF 预测，每折只用自己的训练病人统计标准化参数；最终模型用全部训练集，负责验证集和测试集。窗口继承记录标签，用不加权交叉熵；双导联有 25% 样本只给 1–2 秒 V1 历史。按验证集 loss 早停（最多 50 轮），设置不变时直接复用已训好的模型。

In [8]:
def _resnet_array_hash(digest, array):
    array = np.asarray(array)
    digest.update(str((array.shape, array.dtype)).encode())
    if array.dtype.hasobject:
        digest.update(json.dumps(array.tolist(), ensure_ascii=False).encode())
        return
    for start in range(0, len(array), 1024):
        digest.update(np.ascontiguousarray(array[start:start + 1024]))


def _resnet_signature(signals, labels, patient_ids, record_ids, raw_v1, preprocessing, config):
    digest = hashlib.sha256()
    digest.update(json.dumps(config, sort_keys=True).encode())
    for split in ("train", "val"):
        for array in (signals[split], labels[split], patient_ids[split]):
            _resnet_array_hash(digest, array)
        if record_ids is not None:
            _resnet_array_hash(digest, record_ids[split])
    if raw_v1 is not None:
        _resnet_array_hash(digest, raw_v1)
    if preprocessing is not None:
        digest.update(json.dumps(preprocessing, sort_keys=True, default=lambda x: x.tolist()).encode())
    return digest.hexdigest()


def _resnet_atomic_save(checkpoint, path):
    with NamedTemporaryFile(dir=path.parent, prefix=path.name + ".", delete=False) as temp:
        temporary_path = Path(temp.name)
    try:
        torch.save(checkpoint, temporary_path)
        temporary_path.replace(path)
    finally:
        temporary_path.unlink(missing_ok=True)


def load_resnet_checkpoint(path, device="cpu"):
    """Return the frozen network and its optional foldwise channel normalization."""
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    model = CausalResNet18(checkpoint["channels"], kernel_size=checkpoint["kernel_size"])
    model.load_state_dict(checkpoint["state_dict"])
    model.to(device).eval()
    moments = checkpoint["fold_scale"]
    fold_scale = None if moments is None else tuple(np.asarray(value, dtype=np.float32) for value in moments)
    return model, fold_scale

In [9]:
def _resnet_fit(
    train_windows, val_windows, train_labels, val_labels, fit_rows, channels, fold_scale, device,
    checkpoint_path, signature, seed, config, wake_source
):
    # Include the actual fit partition so a different GroupKFold assignment invalidates this model.
    fold_digest = hashlib.sha256(signature.encode())
    _resnet_array_hash(fold_digest, np.asarray(fit_rows, dtype=np.int64))
    signature = fold_digest.hexdigest()
    if checkpoint_path.is_file():
        try:
            checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
        except (OSError, EOFError, RuntimeError, UnpicklingError) as error:
            print(f"{checkpoint_path.stem}: rebuilding unreadable checkpoint ({error})", flush=True)
            checkpoint = {}
        if isinstance(checkpoint, dict) and checkpoint.get("signature") == signature:
            model = CausalResNet18(channels, config["kernel_size"]).to(device)
            model.load_state_dict(checkpoint["state_dict"])
            print(
                f"{checkpoint_path.stem}: reusing epoch {checkpoint['epoch']}, "
                f"val loss {checkpoint['val_loss']:.5f}", flush=True
            )
            return model.eval()

    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = CausalResNet18(channels, config["kernel_size"]).to(device)
    parameter_count = sum(parameter.numel() for parameter in model.parameters())
    if not 400_000 <= parameter_count <= 1_500_000:
        raise RuntimeError(f"Unexpected ResNet parameter count: {parameter_count}")
    optimizer = torch.optim.AdamW(model.parameters(), lr=config["learning_rate"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=config["lr_patience"], threshold=0,
        min_lr=config["learning_rate"] * 0.01,
    )
    best_epoch = 0
    best_loss = float("inf")
    history = []
    val_steps = np.asarray(config["validation_steps"]) - 1
    windows_per_record = config["windows_per_record"]
    n_examples = len(fit_rows) * windows_per_record
    # Random order per record, fixed over the run: each ten consecutive slots cover all ten seconds.
    step_schedule = rng.random((len(fit_rows), RESNET_STEPS)).argsort(axis=1)

    for epoch in range(1, config["max_epochs"] + 1):
        model.train()
        train_batch_size = config["batch_size"]
        train_loss = torch.zeros((), device=device)
        current_lr = optimizer.param_groups[0]["lr"]
        ordering = rng.permutation(n_examples)
        for start in range(0, n_examples, train_batch_size):
            slots = ordering[start:start + train_batch_size]
            positions = slots // windows_per_record
            rows = fit_rows[positions]
            time_slots = (epoch - 1) * windows_per_record + slots % windows_per_record
            steps = step_schedule[positions, time_slots % RESNET_STEPS]
            batch = _resnet_batch(train_windows, rows, steps, channels, fold_scale, wake_source, rng)
            truth = torch.from_numpy(train_labels[rows]).to(device)
            optimizer.zero_grad(set_to_none=True)
            # Accumulate gradients in float32. The optional CUDA autocast covers training only.
            precision = (torch.autocast("cuda", dtype=torch.bfloat16)
                         if config["cuda_bfloat16"] else nullcontext())
            with precision:
                logits = model(torch.from_numpy(batch).to(device))
                loss = F.cross_entropy(logits, truth)
            loss.backward()
            optimizer.step()
            train_loss += loss.detach() * len(rows)

        train_loss_value = float(train_loss.item()) / n_examples
        val_loss = _resnet_validation_loss(
            model, val_windows, val_labels, val_steps, channels, fold_scale, device,
            config["eval_batch_size"],
        )
        if not np.isfinite([train_loss_value, val_loss]).all():
            raise RuntimeError(f"{checkpoint_path.stem} produced a non-finite loss in epoch {epoch}")
        history.append({"epoch": epoch, "train_loss": train_loss_value, "val_loss": val_loss,
                        "learning_rate": current_lr})
        scheduler.step(val_loss)
        print(
            f"{checkpoint_path.stem} epoch {epoch:02d}/{config['max_epochs']:02d} "
            f"train_loss={train_loss_value:.5f} val_loss={val_loss:.5f} lr={current_lr:.2g}", flush=True
        )
        if val_loss < best_loss:
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
            best_loss = val_loss
            best_epoch = epoch
        if epoch - best_epoch >= config["patience"]:
            break

    checkpoint = {
        "state_dict": best_state,
        "channels": channels,
        "kernel_size": config["kernel_size"],
        "config": config,
        "model_seed": seed,
        "signature": signature,
        "parameter_count": parameter_count,
        "epoch": best_epoch,
        "val_loss": best_loss,
        "history": history,
        "validation_steps": config["validation_steps"],
        "fold_scale": None if fold_scale is None else [value.tolist() for value in fold_scale],
    }
    _resnet_atomic_save(checkpoint, checkpoint_path)
    model.load_state_dict(best_state)
    print(f"{checkpoint_path.stem}: selected epoch {best_epoch}, val loss {best_loss:.5f}", flush=True)
    return model.eval()

In [10]:
def train_resnets(
    stream_inputs, labels, patient_ids, device, out_dir, *, record_ids=None, raw_v1=None, preprocessing=None,
    wake_window_builder=None, partial_wake_fraction=0.25, max_epochs=50, patience=9,
    batch_size=256, eval_batch_size=512, learning_rate=1e-3, lr_patience=3, windows_per_record=3,
    kernel_size=11, validation_steps=tuple(range(1, 11)), cuda_bfloat16=True, seed=42,
):
    """Fit independent single/dual models and return per-step OOF, validation and test posteriors."""
    out_dir = Path(out_dir)
    fold_dir = out_dir / "fold_models"
    fold_dir.mkdir(parents=True, exist_ok=True)
    device = torch.device(device)
    validation_steps = tuple(int(step) for step in validation_steps)
    if not validation_steps or not set(validation_steps) <= set(range(1, RESNET_STEPS + 1)):
        raise ValueError("Validation steps must be integers from 1 through 10")
    if min(max_epochs, patience, batch_size, eval_batch_size) < 1 or not 0 <= partial_wake_fraction <= 1:
        raise ValueError("Invalid training schedule or partial V1 fraction")
    if not isinstance(windows_per_record, (int, np.integer)) or not 1 <= windows_per_record <= RESNET_STEPS:
        raise ValueError("windows_per_record must be an integer from 1 through 10")
    if not np.isfinite(learning_rate) or learning_rate <= 0 or lr_patience < 0:
        raise ValueError("Pass a positive learning rate and nonnegative scheduler patience")
    use_bfloat16 = bool(cuda_bfloat16 and device.type == "cuda" and torch.cuda.is_bf16_supported(including_emulation=False))
    config = dict(
        version=RESNET_CACHE_VERSION, max_epochs=max_epochs, patience=patience, batch_size=batch_size,
        eval_batch_size=eval_batch_size, learning_rate=learning_rate, lr_patience=lr_patience,
        windows_per_record=int(windows_per_record), kernel_size=kernel_size, cuda_bfloat16=use_bfloat16,
        validation_steps=validation_steps, seed=seed, partial_wake_fraction=partial_wake_fraction,
        device_type=device.type, torch_version=str(torch.__version__),
    )
    signals = {split: np.asarray(stream_inputs[split], dtype=np.float32) for split in ("train", "val", "test")}
    for split, signal in signals.items():
        if signal.ndim != 3 or signal.shape[1:] != (2, RESNET_STEPS * RESNET_RATE) or len(signal) == 0:
            raise ValueError(f"{split} must have shape [N, 2, 1000] and be nonempty")

    train_labels = np.asarray(labels["train"], dtype=np.int64)
    val_labels = np.asarray(labels["val"], dtype=np.int64)
    for split, target in (("train", train_labels), ("val", val_labels)):
        if target.shape != (len(signals[split]),) or not np.isin(target, np.arange(4)).all():
            raise ValueError(f"{split} must have aligned integer class indices from 0 through 3")
    patients_train = np.asarray(patient_ids["train"])
    patients_val = np.asarray(patient_ids["val"])
    seen = np.empty(0, dtype=patients_train.dtype)
    for split in ("train", "val", "test"):
        patients = np.asarray(patient_ids[split])
        if patients.shape != (len(signals[split]),) or np.intersect1d(seen, patients).size:
            raise ValueError(f"{split} patient IDs are unaligned or overlap another official split")
        seen = np.union1d(seen, patients)

    if record_ids is not None:
        for split, signal in signals.items():
            if np.asarray(record_ids[split]).shape != (len(signal),):
                raise ValueError(f"{split} ECG IDs do not align with the waveform rows")

    wake_source = None
    if raw_v1 is not None and partial_wake_fraction > 0:
        raw_v1 = raw_v1["train"] if isinstance(raw_v1, dict) else raw_v1
        raw_v1 = np.asarray(raw_v1, dtype=np.float32)
        if raw_v1.shape != signals["train"][:, 1].shape or preprocessing is None:
            raise ValueError("Partial V1 windows require aligned raw train V1 and preprocessing parameters")
        if wake_window_builder is None:
            wake_window_builder = make_wake_windows
        wake_source = dict(
            raw_v1=raw_v1, preprocessing=preprocessing, builder=wake_window_builder,
            fraction=partial_wake_fraction, lead_i=signals["train"][:, 0],
        )

    signature = _resnet_signature(
        signals, {"train": train_labels, "val": val_labels},
        {"train": patients_train, "val": patients_val},
        record_ids, None if wake_source is None else raw_v1, preprocessing, config,
    )
    windows = {split: _resnet_windows(signal) for split, signal in signals.items()}
    folds = list(GroupKFold(n_splits=5).split(
        np.arange(len(train_labels)), train_labels, groups=patients_train
    ))
    fold_assignments = np.zeros(len(train_labels), dtype=np.int64)
    fold_scales = []
    for fold, (fit_rows, held_out_rows) in enumerate(folds, start=1):
        fit_patients = patients_train[fit_rows]
        held_out_patients = patients_train[held_out_rows]
        if np.intersect1d(fit_patients, held_out_patients).size or fold_assignments[held_out_rows].any():
            raise RuntimeError("A patient or record crosses an OOF fit/held-out partition")
        fold_assignments[held_out_rows] = fold
        fold_scales.append(_resnet_fold_scale(signals["train"], fit_rows, preprocessing))
    if not fold_assignments.all():
        raise RuntimeError("Some training records have no OOF assignment")

    output = {}
    for branch, channels, final_name in (("single", 1, "model_lead1.pt"), ("dual", 2, "model_dual.pt")):
        train_predictions = np.empty((len(train_labels), RESNET_STEPS, 4), dtype=np.float32)
        branch_wake_source = wake_source if channels == 2 else None
        for index, ((fit_rows, held_out_rows), fold_scale) in enumerate(zip(folds, fold_scales), start=1):
            model = _resnet_fit(
                windows["train"], windows["val"], train_labels, val_labels, fit_rows, channels,
                fold_scale, device, fold_dir / f"{branch}_fold{index}.pt",
                f"{signature}:{branch}:{index}", seed + channels * 100 + index, config,
                branch_wake_source,
            )
            train_predictions[held_out_rows] = _resnet_probabilities(
                model, windows["train"], held_out_rows, channels, fold_scale, device, eval_batch_size
            )

        final_model = _resnet_fit(
            windows["train"], windows["val"], train_labels, val_labels,
            np.arange(len(train_labels)), channels, None, device, out_dir / final_name,
            f"{signature}:{branch}:final", seed + channels * 100, config, branch_wake_source,
        )
        output[branch] = {"train": train_predictions}
        for split in ("val", "test"):
            output[branch][split] = _resnet_probabilities(
                final_model, windows[split], np.arange(len(windows[split])), channels, None,
                device, eval_batch_size,
            )
    np.save(out_dir / "train_oof_fold.npy", fold_assignments, allow_pickle=False)
    return output

### 训练并导出逐秒概率

输出 `obs_probs_{single,dual}_{split}.npy`，形状 `[N, 10, 4]`。

In [11]:
streams, raw_v1, preprocessing = build_stream_inputs()
probabilities = train_resnets(
    streams, labels, patient_ids, device, OUTPUT_DIR,
    record_ids=ids, raw_v1=raw_v1, preprocessing=preprocessing,
    wake_window_builder=make_wake_windows, **RESNET_OPTIONS,
)
for branch, by_split in probabilities.items():
    for split, values in by_split.items():
        target = OUTPUT_DIR / f"obs_probs_{branch}_{split}.npy"
        partial = target.with_suffix(".partial.npy")
        np.save(partial, values, allow_pickle=False)
        partial.replace(target)

single_fold1 epoch 01/50 train_loss=0.97229 val_loss=0.89059 lr=0.001
single_fold1 epoch 02/50 train_loss=0.82701 val_loss=1.00392 lr=0.001
single_fold1 epoch 03/50 train_loss=0.78858 val_loss=0.89005 lr=0.001
single_fold1 epoch 04/50 train_loss=0.76768 val_loss=0.76945 lr=0.001
single_fold1 epoch 05/50 train_loss=0.75564 val_loss=0.91523 lr=0.001
single_fold1 epoch 06/50 train_loss=0.74337 val_loss=0.77100 lr=0.001
single_fold1 epoch 07/50 train_loss=0.73362 val_loss=0.82456 lr=0.001
single_fold1 epoch 08/50 train_loss=0.72518 val_loss=0.73945 lr=0.001
single_fold1 epoch 09/50 train_loss=0.71943 val_loss=0.73368 lr=0.001
single_fold1 epoch 10/50 train_loss=0.70965 val_loss=0.75503 lr=0.001
single_fold1 epoch 11/50 train_loss=0.70639 val_loss=0.75642 lr=0.001
single_fold1 epoch 12/50 train_loss=0.70478 val_loss=0.75805 lr=0.001
single_fold1 epoch 13/50 train_loss=0.69781 val_loss=0.74806 lr=0.001
single_fold1 epoch 14/50 train_loss=0.67441 val_loss=0.73584 lr=0.0005
single_fold1 epoch 

## 4. Medformer 基准（G5）

输入 12 导联整段 10 秒，3 种 patch 长度（20 / 50 / 100）、3 层、宽度 128；按验证集 Macro-F1 挑模型，最多 60 轮。参考：[论文](https://proceedings.neurips.cc/paper_files/paper/2024/hash/3fe2a777282299ecb4f9e7ebb531f0ab-Abstract-Conference.html) · [作者实现](https://github.com/DL4mHealth/Medformer)。

In [12]:
# %% Medformer architecture
# Wang et al., NeurIPS 2024, Section 4 (cross-channel patching and routed attention).
# https://papers.nips.cc/paper_files/paper/2024/file/3fe2a777282299ecb4f9e7ebb531f0ab-Paper-Conference.pdf
# https://github.com/DL4mHealth/Medformer (official router-only classification head)
# This adaptation consumes full 10 s / 100 Hz, 12-lead recordings and
# has a four-class head. It uses the paper's explicit position + granularity
# routers and the authors' updated router-only readout. No augmentation is used.


class _MedformerEncoderLayer(nn.Module):
    """Per-scale attention followed by communication among scale routers."""

    def __init__(self, granularities, width, heads, dropout):
        super().__init__()
        self.intra_attention = nn.ModuleList(
            nn.MultiheadAttention(width, heads, dropout=dropout, batch_first=True)
            for _ in range(granularities)
        )
        self.inter_attention = nn.MultiheadAttention(
            width, heads, dropout=dropout, batch_first=True
        )
        self.attention_norm = nn.LayerNorm(width)
        self.output_norm = nn.LayerNorm(width)
        self.feed_forward = nn.Sequential(
            nn.Linear(width, 2 * width),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(2 * width, width),
        )
        self.dropout = nn.Dropout(dropout)

    def forward(self, scales):
        intra = [
            attention(tokens, tokens, tokens, need_weights=False)[0]
            for attention, tokens in zip(self.intra_attention, scales)
        ]
        routers = torch.stack([tokens[:, -1, :] for tokens in intra], dim=1)
        routers, _ = self.inter_attention(routers, routers, routers, need_weights=False)
        attended = torch.cat(
            [
                torch.cat((tokens[:, :-1], routers[:, index : index + 1]), dim=1)
                for index, tokens in enumerate(intra)
            ],
            dim=1,
        )
        tokens = self.attention_norm(torch.cat(scales, dim=1) + self.dropout(attended))
        tokens = self.output_norm(tokens + self.dropout(self.feed_forward(tokens)))
        lengths = [scale.size(1) for scale in scales]
        return tokens.split(lengths, dim=1)


class Medformer(nn.Module):
    """Full-record G5 baseline: [batch, 12, 1000] -> four logits."""

    def __init__(self, patch_lengths=(20, 50, 100), width=128, heads=8, layers=3, dropout=0.15):
        super().__init__()
        self.input_leads = 12
        self.input_samples = 1000
        if len(set(patch_lengths)) < 2 or any(
            not 1 <= patch <= self.input_samples for patch in patch_lengths
        ):
            raise ValueError("Medformer requires multiple valid patch scales.")
        if heads < 1 or width < 2 or width % math.lcm(heads, 2) or layers < 1:
            raise ValueError("Use an even width divisible by the head count and at least one layer.")

        self.patch_lengths = tuple(patch_lengths)
        self.paddings = tuple(-self.input_samples % patch for patch in self.patch_lengths)
        self.patch_projections = nn.ModuleList(
            nn.Linear(self.input_leads * patch, width, bias=False)
            for patch in self.patch_lengths
        )
        max_patches = math.ceil(self.input_samples / min(self.patch_lengths))
        steps = torch.arange(max_patches + 1, dtype=torch.float32).unsqueeze(1)
        frequencies = torch.exp(torch.arange(0, width, 2) * (-math.log(10000.0) / width))
        position = torch.empty(max_patches + 1, width)
        position[:, 0::2] = torch.sin(steps * frequencies)
        position[:, 1::2] = torch.cos(steps * frequencies)
        self.register_buffer("position", position)
        self.granularity = nn.Parameter(torch.empty(len(self.patch_lengths), width))
        nn.init.normal_(self.granularity, std=0.02)
        self.embedding_dropout = nn.Dropout(dropout)
        self.layers = nn.ModuleList(
            _MedformerEncoderLayer(len(self.patch_lengths), width, heads, dropout)
            for _ in range(layers)
        )
        self.classifier = nn.Sequential(
            nn.LayerNorm(width),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Flatten(start_dim=1),
            nn.Linear(len(self.patch_lengths) * width, 4),
        )

    def forward(self, signal):
        if signal.ndim != 3 or signal.shape[1:] != (self.input_leads, self.input_samples):
            raise ValueError(f"Expected [B, 12, 1000], got {tuple(signal.shape)}.")

        scales = []
        for index, (patch, padding, project) in enumerate(
            zip(self.patch_lengths, self.paddings, self.patch_projections)
        ):
            padded = F.pad(signal, (0, padding)) if padding else signal
            # Each token projects all 12 leads over the same temporal patch.
            patches = padded.unfold(-1, patch, patch).transpose(1, 2).flatten(start_dim=2)
            content = project(patches)
            # Content-free router initialized by the next position and scale embedding.
            router = content.new_zeros(content.size(0), 1, content.size(-1))
            tokens = torch.cat((content, router), dim=1)
            tokens = tokens + self.position[: tokens.size(1)] + self.granularity[index]
            scales.append(self.embedding_dropout(tokens))

        for layer in self.layers:
            scales = layer(scales)
        routers = torch.stack([tokens[:, -1, :] for tokens in scales], dim=1)
        return self.classifier(routers)

In [13]:
# %% Training and offline G5 prediction


MEDFORMER_HIGH_CONFIG = {
    "patch_lengths": [20, 50, 100],
    "width": 128,
    "heads": 8,
    "layers": 3,
    "dropout": 0.15,
}
MEDFORMER_FILES = (
    "model_medformer_12lead.pt",
    "preds_g5_medformer_val.npy",
    "preds_g5_medformer_test.npy",
)


def medformer_run_is_complete(out_dir, signature):
    """Verify that the three saved artifacts belong to the signed training run."""
    out_dir = Path(out_dir)
    try:
        info = json.loads((out_dir / "medformer_training_history.json").read_text(encoding="utf-8"))
        return bool(signature) and info.get("input_signature") == signature and all(
            hashlib.sha256((out_dir / name).read_bytes()).hexdigest() == info["sha256"][name]
            for name in MEDFORMER_FILES
        )
    except (OSError, ValueError, KeyError, TypeError, AttributeError):
        return False


def load_medformer_checkpoint(out_dir, device="cpu"):
    """Load a compact or expanded G5 state dict with that run's architecture."""
    out_dir = Path(out_dir)
    info = json.loads((out_dir / "medformer_training_history.json").read_text(encoding="utf-8"))
    model = Medformer(**info["model_kwargs"])
    state = torch.load(out_dir / MEDFORMER_FILES[0], map_location="cpu", weights_only=True)
    model.load_state_dict(state)
    return model.to(device).eval()


def _medformer_batch(array, rows, device, dtype):
    # Advanced indexing already copies. Copy a read-only slice before wrapping it.
    batch = np.asarray(array[rows], dtype=dtype)
    if not batch.flags.writeable:
        batch = batch.copy()
    return torch.from_numpy(batch).to(device)


@torch.inference_mode()
def _medformer_probabilities(model, records, device, batch_size):
    model.eval()
    probabilities = np.empty((len(records), 4), dtype=np.float32)
    for start in range(0, len(records), batch_size):
        rows = slice(start, start + batch_size)
        x = _medformer_batch(records, rows, device, np.float32)
        probabilities[rows] = model(x).softmax(dim=-1).cpu().numpy()
    if not np.isfinite(probabilities).all():
        raise FloatingPointError("Medformer produced nonfinite probabilities.")
    return probabilities


def train_medformer(
    inputs,
    labels,
    device,
    out_dir,
    *,
    max_epochs=60,
    patience=12,
    batch_size=96,
    learning_rate=3e-4,
    seed=42,
    model_kwargs=None,
    input_signature=None,
):
    """Train on all train rows, select by validation macro-F1, and score test waveforms.

    Input arrays are keyed by train/val/test and stay memory-mapped. Labels use
    0=NORM, 1=AFIB, 2=MI, 3=STTC. Only train and val labels are accessed.
    The returned information includes per-epoch metrics and constructor options.
    """
    if min(max_epochs, patience, batch_size) < 1 or learning_rate <= 0:
        raise ValueError("Epochs, patience, batch size and learning rate must be positive.")
    for split in ("train", "val", "test"):
        records = inputs[split]
        if records.ndim != 3 or records.shape[1:] != (12, 1000) or len(records) == 0:
            raise ValueError(f"Expected a nonempty {split} array of shape [N, 12, 1000].")
    target = {}
    for split in ("train", "val"):
        encoded = np.asarray(labels[split])
        if (
            encoded.shape != (len(inputs[split]),)
            or not np.issubdtype(encoded.dtype, np.integer)
            or not np.all((0 <= encoded) & (encoded < 4))
        ):
            raise ValueError(f"Expected one integer class index per {split} recording.")
        target[split] = encoded

    device = torch.device(device)
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    config = {**MEDFORMER_HIGH_CONFIG, **(model_kwargs or {})}
    model = Medformer(**config).to(device)
    weight_decay = 1e-2
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    schedule = dict(mode="max", factor=0.5, patience=3, threshold_mode="abs",
                    threshold=0.0, min_lr=min(1e-6, learning_rate))
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, **schedule)
    criterion = nn.CrossEntropyLoss()
    n_train = len(inputs["train"])
    best_f1 = -1.0
    best_epoch = 0
    best_weights = None
    best_val_probabilities = None
    history = []

    for epoch in range(1, max_epochs + 1):
        started = time.perf_counter()
        epoch_lr = optimizer.param_groups[0]["lr"]
        model.train()
        permutation = rng.permutation(n_train)
        loss_sum = torch.zeros((), device=device)
        for start in range(0, n_train, batch_size):
            rows = permutation[start : start + batch_size]
            x = _medformer_batch(inputs["train"], rows, device, np.float32)
            y = _medformer_batch(target["train"], rows, device, np.int64)
            optimizer.zero_grad(set_to_none=True)
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            loss_sum += loss.detach() * len(rows)

        train_loss = float(loss_sum.item() / n_train)
        if not np.isfinite(train_loss):
            raise FloatingPointError("Medformer training loss became nonfinite.")
        val_probabilities = _medformer_probabilities(model, inputs["val"], device, batch_size)
        val_f1 = float(
            f1_score(
                target["val"], val_probabilities.argmax(axis=1),
                labels=(0, 1, 2, 3), average="macro", zero_division=0,
            )
        )
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch
            best_val_probabilities = val_probabilities
            best_weights = {
                name: value.detach().to(device="cpu", copy=True)
                for name, value in model.state_dict().items()
            }
        scheduler.step(val_f1)
        seconds = time.perf_counter() - started
        history.append(
            {"epoch": epoch, "train_loss": train_loss, "val_macro_f1": val_f1,
             "learning_rate": epoch_lr, "seconds": seconds}
        )
        print(
            f"Medformer G5 epoch {epoch:02d}/{max_epochs}: "
            f"loss={train_loss:.4f}, val macro-F1={val_f1:.4f}, lr={epoch_lr:.2g}, {seconds:.1f}s",
            flush=True,
        )
        if epoch - best_epoch >= patience:
            break

    model.load_state_dict(best_weights)
    test_probabilities = _medformer_probabilities(model, inputs["test"], device, batch_size)
    info = {
        "selection_metric": "val_macro_f1",
        "best_epoch": best_epoch,
        "best_val_macro_f1": best_f1,
        "history": history,
        "model_kwargs": config,
        "parameters": sum(parameter.numel() for parameter in model.parameters()),
        "seed": seed,
        "device": str(device),
        "batch_size": batch_size,
        "max_epochs": max_epochs,
        "patience": patience,
        "optimizer": {
            "name": "AdamW",
            "learning_rate": learning_rate,
            "weight_decay": weight_decay,
            "loss": "cross_entropy",
        },
        "lr_scheduler": {"name": "ReduceLROnPlateau", **schedule},
        "class_order": ["NORM", "AFIB", "MI", "STTC"],
        "precision": "float32",
        "input_signature": input_signature,
        "checkpoint_path": MEDFORMER_FILES[0],
        "val_prediction_path": MEDFORMER_FILES[1],
        "prediction_path": MEDFORMER_FILES[2],
        "history_path": "medformer_training_history.json",
    }
    # Stage the whole group before publishing. The manifest is the commit marker.
    with TemporaryDirectory(dir=out_dir, prefix=".medformer-") as temp:
        staging = Path(temp)
        torch.save(best_weights, staging / MEDFORMER_FILES[0])
        np.save(staging / MEDFORMER_FILES[1], best_val_probabilities, allow_pickle=False)
        np.save(staging / MEDFORMER_FILES[2], test_probabilities, allow_pickle=False)
        info["sha256"] = {
            name: hashlib.sha256((staging / name).read_bytes()).hexdigest() for name in MEDFORMER_FILES
        }
        name = info["history_path"]
        (staging / name).write_text(json.dumps(info, indent=2, allow_nan=False) + "\n", encoding="utf-8")
        for filename in (*MEDFORMER_FILES, name):
            (staging / filename).replace(out_dir / filename)
    return info

In [14]:
# Reuse a completed training run only when the inputs and training settings match.
med_options = MEDFORMER_OPTIONS
digest = hashlib.sha256()
digest.update(json.dumps({"version": 2, "options": med_options, "torch": str(torch.__version__),
                          "device": device.type, "classes": CLASSES}, sort_keys=True).encode())
for split in SPLITS:
    _resnet_array_hash(digest, offline_inputs[split])
    _resnet_array_hash(digest, ids[split])
    if split in labels:
        _resnet_array_hash(digest, labels[split])
med_signature = digest.hexdigest()
history_path = OUTPUT_DIR / "medformer_training_history.json"
if medformer_run_is_complete(OUTPUT_DIR, med_signature):
    history = json.loads(history_path.read_text(encoding="utf-8"))
else:
    history = train_medformer(offline_inputs, labels, device, OUTPUT_DIR,
                              input_signature=med_signature, **med_options)
print(f"Medformer selected epoch {history['best_epoch']}, validation macro-F1 {history['best_val_macro_f1']:.4f}.")

Medformer G5 epoch 01/60: loss=0.9192, val macro-F1=0.5209, lr=0.0003, 120.7s
Medformer G5 epoch 02/60: loss=0.6948, val macro-F1=0.5775, lr=0.0003, 76.2s
Medformer G5 epoch 03/60: loss=0.6181, val macro-F1=0.6253, lr=0.0003, 76.3s
Medformer G5 epoch 04/60: loss=0.5518, val macro-F1=0.6941, lr=0.0003, 76.7s
Medformer G5 epoch 05/60: loss=0.5119, val macro-F1=0.6812, lr=0.0003, 76.6s
Medformer G5 epoch 06/60: loss=0.4688, val macro-F1=0.6852, lr=0.0003, 76.8s
Medformer G5 epoch 07/60: loss=0.4313, val macro-F1=0.6798, lr=0.0003, 76.0s
Medformer G5 epoch 08/60: loss=0.4023, val macro-F1=0.6952, lr=0.0003, 76.5s
Medformer G5 epoch 09/60: loss=0.3882, val macro-F1=0.6884, lr=0.0003, 76.8s
Medformer G5 epoch 10/60: loss=0.3507, val macro-F1=0.6957, lr=0.0003, 76.4s
Medformer G5 epoch 11/60: loss=0.3335, val macro-F1=0.7039, lr=0.0003, 76.8s
Medformer G5 epoch 12/60: loss=0.3139, val macro-F1=0.6806, lr=0.0003, 76.4s
Medformer G5 epoch 13/60: loss=0.2955, val macro-F1=0.6984, lr=0.0003, 75.6

## 5. 核对输出与验证集结果

检查概率文件的形状和数值，导出 `record_index.csv`；在验证集上比较 ResNet（10 秒平均）和 Medformer 的 Macro-F1 / AUROC。

In [15]:
def read_probabilities(filename, expected_shape):
    values = np.load(OUTPUT_DIR / filename, allow_pickle=False)
    if values.shape != expected_shape or values.dtype != np.float32:
        raise ValueError(f"Invalid shape or dtype in {filename}")
    if not np.isfinite(values).all() or values.min() < 0 or values.max() > 1:
        raise ValueError(f"Invalid probabilities in {filename}")
    np.testing.assert_allclose(values.sum(axis=-1), 1, rtol=0, atol=2e-5)
    return values


folds = np.load(OUTPUT_DIR / "train_oof_fold.npy", allow_pickle=False)
if folds.shape != (len(ids["train"]),) or len(np.unique(folds)) != 5:
    raise ValueError("Expected five OOF assignments for the selected training records")
fold_map = pd.DataFrame({"patient_id": patient_ids["train"], "oof_fold": folds})
if (fold_map.groupby("patient_id")["oof_fold"].nunique() != 1).any():
    raise ValueError("OOF assignments split records belonging to the same patient")

index_tables = []
for split in SPLITS:
    index_tables.append(pd.DataFrame({"split": split, "row": np.arange(len(ids[split])),
                                     "ecg_id": ids[split], "patient_id": patient_ids[split],
                                     "oof_fold": folds if split == "train" else 0}))
    for branch in ("single", "dual"):
        read_probabilities(f"obs_probs_{branch}_{split}.npy", (len(ids[split]), 10, 4))
pd.concat(index_tables, ignore_index=True).to_csv(OUTPUT_DIR / "record_index.csv", index=False)
read_probabilities("preds_g5_medformer_test.npy", (len(ids["test"]), 4))

val_scores = {}
for branch in ("single", "dual"):
    # Report a record-level mean of the ten causal outputs for comparison on validation.
    probs = read_probabilities(f"obs_probs_{branch}_val.npy", (len(ids["val"]), 10, 4))
    val_scores[f"ResNet {branch} (10-step mean)"] = probs.mean(axis=1)
val_scores["Medformer (12 leads, 10 s)"] = read_probabilities(
    "preds_g5_medformer_val.npy", (len(ids["val"]), 4)
)
rows = []
for model_name, probs in val_scores.items():
    rows.append({
        "model": model_name,
        "macro_f1": f1_score(labels["val"], probs.argmax(axis=1), average="macro", zero_division=0),
        "macro_auroc_ovr": roc_auc_score(labels["val"], probs, multi_class="ovr", average="macro"),
    })
validation_metrics = pd.DataFrame(rows)
validation_metrics.to_csv(OUTPUT_DIR / "validation_metrics.csv", index=False)
print(validation_metrics.round(4).to_string(index=False))

training_rows = []
for model_path in sorted((OUTPUT_DIR / "fold_models").glob("*_fold*.pt")) + [
    OUTPUT_DIR / "model_lead1.pt", OUTPUT_DIR / "model_dual.pt"
]:
    saved = torch.load(model_path, map_location="cpu", weights_only=True)
    for epoch in saved["history"]:
        training_rows.append({"model": model_path.stem, "selected": epoch["epoch"] == saved["epoch"],
                              "parameters": saved["parameter_count"], **epoch})
pd.DataFrame(training_rows).to_csv(OUTPUT_DIR / "resnet_training_history.csv", index=False)

                       model  macro_f1  macro_auroc_ovr
ResNet single (10-step mean)    0.6661           0.8891
  ResNet dual (10-step mean)    0.7012           0.9051
  Medformer (12 leads, 10 s)    0.7056           0.9208


## 6. V1 唤醒回放

`RawECGReplay.predict(ecg_id, end_step, wake_step)` 只读取截至当前秒的原始信号，返回该秒的 4 类概率；train 记录自动用所在折的模型。V1 中途唤醒后，每一秒都要从唤醒时刻重新滤波。

In [16]:
class RawECGReplay:
    """Read only visible samples and select the correct frozen ResNet by ECG ID."""

    def __init__(self, data_dir, output_dir, device="cpu"):
        self.data_dir = Path(data_dir)
        self.output_dir = Path(output_dir)
        self.device = torch.device(device)
        self.index = pd.read_csv(self.output_dir / "record_index.csv").set_index("ecg_id", verify_integrity=True)
        self.paths = pd.read_csv(self.data_dir / "ptbxl_database.csv", usecols=["ecg_id", "filename_lr"]
                                 ).set_index("ecg_id", verify_integrity=True)["filename_lr"]
        self.preprocessing = json.loads((self.output_dir / "stream_preprocessing.json").read_text())
        self.sos = np.array(self.preprocessing["sos"])
        self.models = {}

    def model_for(self, branch, fold):
        if (branch, fold) not in self.models:
            if fold:
                file = self.output_dir / "fold_models" / f"{branch}_fold{fold}.pt"
            else:
                file = self.output_dir / ("model_lead1.pt" if branch == "single" else "model_dual.pt")
            self.models[branch, fold] = load_resnet_checkpoint(file, self.device)
        return self.models[branch, fold]

    def observed_lead(self, path, channel, start_step, end_step):
        raw, header = wfdb.rdsamp(
            str(path), channel_names=[STREAM_LEADS[channel]],
            sampfrom=start_step * STREAM_SAMPLE_RATE, sampto=end_step * STREAM_SAMPLE_RATE,
        )
        expected = (end_step - start_step) * STREAM_SAMPLE_RATE
        if (header["fs"] != STREAM_SAMPLE_RATE or raw.shape != (expected, 1)
                or header["sig_name"] != [STREAM_LEADS[channel]] or header["units"] != ["mV"]):
            raise ValueError(f"Invalid stream segment in {path}")
        if not np.isfinite(raw).all():
            raise ValueError(f"Non-finite waveform in {path}")
        filtered = filter_stream(raw[:, 0].astype(np.float32), self.sos).astype(np.float32)
        mean = self.preprocessing["mean_mv"][channel]
        scale = self.preprocessing["std_mv"][channel] + self.preprocessing["std_epsilon"]
        observed = ((filtered[-RESNET_WINDOW:].astype(np.float64) - mean) / scale).astype(np.float32)
        return np.pad(observed, (RESNET_WINDOW - len(observed), 0))

    @torch.inference_mode()
    def predict(self, ecg_id, end_step, wake_step=None):
        """Return four probabilities; wake at the end of w first supplies V1 at w+1."""
        if not isinstance(end_step, (int, np.integer)) or not 1 <= end_step <= STREAM_DURATION:
            raise ValueError("end_step must be an integer from 1 to 10")
        if wake_step is not None and (not isinstance(wake_step, (int, np.integer))
                                      or not 0 <= wake_step <= STREAM_DURATION):
            raise ValueError("wake_step must be None or an integer from 0 to 10")
        row = self.index.loc[ecg_id]
        path = self.data_dir / self.paths.loc[ecg_id]
        leads = [self.observed_lead(path, 0, 0, end_step)]
        available = [min(3, end_step)]
        branch = "single"
        if wake_step is not None and wake_step < end_step:
            leads.append(self.observed_lead(path, 1, wake_step, end_step))
            available.append(min(3, end_step - wake_step))
            branch = "dual"
        model, moments = self.model_for(branch, int(row.oof_fold))
        batch = _resnet_apply_fold_scale(np.stack(leads)[None], np.array([available]), moments)
        return model(torch.from_numpy(batch).to(self.device)).softmax(dim=-1)[0].cpu().numpy()

## 7. 验证因果性与唤醒效果

核对回放结果与保存的概率一致、与流程一的 `X` 一致；改动未来信号或唤醒前的 V1，确认输出不变；比较只用 Lead I、V1 全程开启、第 4 秒后唤醒 V1 三种情况。

In [17]:
replay = RawECGReplay(DATA_DIR, OUTPUT_DIR, device)
fold_rows = [np.flatnonzero(folds == k)[0] for k in range(1, 6)]
probe_rows = {"train": fold_rows, "val": [0, len(ids["val"]) // 2, len(ids["val"]) - 1]}
cache_error = 0.0
for split, positions in probe_rows.items():
    for branch, wake in (("single", None), ("dual", 0)):
        saved = np.load(OUTPUT_DIR / f"obs_probs_{branch}_{split}.npy", mmap_mode="r")
        for i in positions:
            for t in (1, 2, 4, 10):
                observed = replay.predict(ids[split][i], t, wake)
                expected = saved[i, t - 1]
                np.testing.assert_allclose(observed, expected, atol=2e-5, rtol=1e-5)
                cache_error = max(cache_error, float(np.abs(observed - expected).max()))

# Recompute the offline Phase 1 representation from raw samples at both ends and in
# the middle of each split. Compare record content as well as the ID/row checks above.
offline_error = 0.0
for split in SPLITS:
    for i in (0, len(ids[split]) // 2, len(ids[split]) - 1):
        raw = read_stream_record(DATA_DIR / database.loc[ids[split][i], "filename_lr"])
        filtered = sosfiltfilt(STREAM_SOS, raw, axis=-1)
        offline = (filtered - filtered.mean(axis=-1, keepdims=True)) / (
            filtered.std(axis=-1, keepdims=True) + phase1["signal"]["std_epsilon"]
        )
        difference = np.abs(offline - offline_inputs[split][i, [0, 6]])
        offline_error = max(offline_error, float(difference.max()))
        np.testing.assert_allclose(offline, offline_inputs[split][i, [0, 6]], rtol=1e-5, atol=1e-5)

@torch.inference_mode()
def val_posteriors(windows, branch, batch_size=256):
    model, scale = replay.model_for(branch, 0)
    if scale is not None:
        raise ValueError("Expected full-training weights for validation")
    result = []
    for start in range(0, len(windows), batch_size):
        x = torch.from_numpy(np.ascontiguousarray(windows[start:start + batch_size])).to(device)
        result.append(model(x).softmax(dim=-1).cpu().numpy())
    return np.concatenate(result)

# Paired interventions on real validation ECGs. The changed samples are deliberately
# large so a leak will not be hidden behind numerical tolerances.
raw = np.stack([read_stream_record(DATA_DIR / database.loc[e, "filename_lr"])
                for e in ids["val"][:16]]).astype(np.float32)
def normalized(raw):
    return standardize_stream(filter_stream(raw).astype(np.float32), preprocessing)

base = normalized(raw)
future = raw.copy()
future[:, :, 500:] += np.random.default_rng(42).normal(0, 2, size=future[:, :, 500:].shape)
future_stream = normalized(future)
future_errors = []
for branch, channels in (("single", 1), ("dual", 2)):
    a = make_causal_windows(base[:, :channels], 5)
    b = make_causal_windows(future_stream[:, :channels], 5)
    np.testing.assert_array_equal(a, b)
    pa, pb = val_posteriors(a, branch), val_posteriors(b, branch)
    np.testing.assert_allclose(pa, pb, atol=2e-6, rtol=0)
    future_errors.append(float(np.abs(pa - pb).max()))

inactive = future[:, 1].copy()
inactive[:, :400] += np.random.default_rng(7).normal(0, 2, size=inactive[:, :400].shape)
a = make_wake_windows(base[:, 0], raw[:, 1], 4, 5, preprocessing)
b = make_wake_windows(future_stream[:, 0], inactive, 4, 5, preprocessing)
np.testing.assert_array_equal(a, b)
np.testing.assert_array_equal(a[:, 1, :200], 0)
pa, pb = val_posteriors(a, "dual"), val_posteriors(b, "dual")
np.testing.assert_allclose(pa, pb, atol=2e-6, rtol=0)
wake_error = float(np.abs(pa - pb).max())
for i in range(3):
    np.testing.assert_allclose(replay.predict(ids["val"][i], 5, 4), pa[i], atol=2e-5, rtol=1e-5)

checks = {
    "ecg_ids_aligned": len(all_ids), "official_folds_verified": True, "patient_oof_folds": 5,
    "offline_raw_comparison_records": 9, "offline_max_abs_error": offline_error,
    "replay_vs_saved_max_abs_probability_error": cache_error,
    "counterfactual_validation_records": len(raw),
    "future_change_max_abs_probability_error": max(future_errors),
    "inactive_v1_and_future_change_max_abs_probability_error": wake_error,
}
(OUTPUT_DIR / "pathway_checks.json").write_text(json.dumps(checks, indent=2) + "\n")
print(json.dumps(checks, indent=2))

{
  "ecg_ids_aligned": 19064,
  "official_folds_verified": true,
  "patient_oof_folds": 5,
  "offline_raw_comparison_records": 9,
  "offline_max_abs_error": 2.3807182536472737e-07,
  "replay_vs_saved_max_abs_probability_error": 2.384185791015625e-07,
  "counterfactual_validation_records": 16,
  "future_change_max_abs_probability_error": 0.0,
  "inactive_v1_and_future_change_max_abs_probability_error": 0.0
}


In [18]:
# Fixed wake after second 4, evaluated at the same decision times in all conditions.
# This isolates activation history on validation records without training an RL policy.
single = np.load(OUTPUT_DIR / "obs_probs_single_val.npy", mmap_mode="r")
dual = np.load(OUTPUT_DIR / "obs_probs_dual_val.npy", mmap_mode="r")
rows = []
for t in (4, 5, 6, 7, 10):
    single_t, dual_t = single[:, t - 1], dual[:, t - 1]
    if t <= 4:
        wake_t = single_t
    else:
        w = make_wake_windows(streams["val"][:, 0], raw_v1["val"], 4, t, preprocessing)
        wake_t = val_posteriors(w, "dual")
    for name, values in (("I only", single_t), ("I + V1 from start", dual_t), ("V1 after 4 s", wake_t)):
        rows.append({
            "second": t, "condition": name,
            "macro_f1": f1_score(labels["val"], values.argmax(axis=1), average="macro", zero_division=0),
            "macro_auroc_ovr": roc_auc_score(labels["val"], values, multi_class="ovr", average="macro"),
            "mean_abs_difference_vs_always_dual": float(np.abs(values - dual_t).mean()),
        })
wake_evaluation = pd.DataFrame(rows)
wake_evaluation.to_csv(OUTPUT_DIR / "wake_validation.csv", index=False)
print(wake_evaluation.pivot(index="second", columns="condition", values="macro_f1").round(4).to_string())

condition  I + V1 from start  I only  V1 after 4 s
second                                            
4                     0.6895  0.6560        0.6560
5                     0.6924  0.6378        0.6621
6                     0.6827  0.6441        0.6825
7                     0.7015  0.6633        0.7022
10                    0.6889  0.6366        0.6883
